In [1]:
!pip install pandas
!pip install numpy
!pip install scikit-learn
!pip install --pre torch torchvision torchaudio --index-url https://download.pytorch.org/whl/nightly/cpu

Looking in indexes: https://download.pytorch.org/whl/nightly/cpu


In [2]:
import kagglehub

path = kagglehub.dataset_download("ted8080/house-prices-and-images-socal")

print("Path to dataset files:", path)


C:\Users\aliim\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Path to dataset files: C:\Users\aliim\.cache\kagglehub\datasets\ted8080\house-prices-and-images-socal\versions\1


In [3]:
import os
import pandas as pd
import numpy as np
import cv2
from sklearn.model_selection import train_test_split

csv_path = os.path.join(path, "socal2.csv")
image_dir = os.path.join(path, "socal_pics/socal_pics") # Check folder name in your path


df = pd.read_csv(csv_path)


def load_images(df, img_dir, target_size=(64, 64)):
    images = []
    for img_id in df['image_id']:

        img_filename = f"{int(img_id)}.png" 
        
        img_full_path = os.path.join(img_dir, img_filename)
        
        img = cv2.imread(img_full_path)
        if img is not None:
            img = cv2.resize(img, target_size)
            images.append(img)
        else:
            images.append(np.zeros((target_size[0], target_size[1], 3), dtype="float32"))
            
    return np.array(images) / 255.0


X_img = load_images(df.head(1000), image_dir)
X_tab = df.head(1000).drop(['price', 'image_id', 'citi'], axis=1).values
y = df.head(1000)['price'].values

In [4]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class MultiModalHouseModel(nn.Module):
    def __init__(self, tabular_input_dim):
        super(MultiModalHouseModel, self).__init__()
        
        self.cnn = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2), # Output: 16 x 32 x 32
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2), # Output: 32 x 16 x 16
            nn.Flatten(),
            nn.Linear(32 * 16 * 16, 16),
            nn.ReLU()
        )
        
        self.mlp = nn.Sequential(
            nn.Linear(tabular_input_dim, 16),
            nn.ReLU(),
            nn.Linear(16, 8),
            nn.ReLU()
        )
        
        self.fc_final = nn.Sequential(
            nn.Linear(16 + 8, 8),
            nn.ReLU(),
            nn.Linear(8, 1) 
        )

    def forward(self, img, tab):
        cnn_out = self.cnn(img)
        mlp_out = self.mlp(tab)
        
        # Glue them together
        combined = torch.cat((cnn_out, mlp_out), dim=1)
        return self.fc_final(combined)

model = MultiModalHouseModel(tabular_input_dim=5) 
print(model)

MultiModalHouseModel(
  (cnn): Sequential(
    (0): Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Flatten(start_dim=1, end_dim=-1)
    (7): Linear(in_features=8192, out_features=16, bias=True)
    (8): ReLU()
  )
  (mlp): Sequential(
    (0): Linear(in_features=5, out_features=16, bias=True)
    (1): ReLU()
    (2): Linear(in_features=16, out_features=8, bias=True)
    (3): ReLU()
  )
  (fc_final): Sequential(
    (0): Linear(in_features=24, out_features=8, bias=True)
    (1): ReLU()
    (2): Linear(in_features=8, out_features=1, bias=True)
  )
)


In [5]:

features = df.head(1000).drop(['price', 'image_id', 'citi'], axis=1, errors='ignore')

X_tab_clean = pd.get_dummies(features).astype(float).values

X_tab_pt = torch.tensor(X_tab_clean, dtype=torch.float32)

print(f"Success! Tabular tensor shape: {X_tab_pt.shape}")

Success! Tabular tensor shape: torch.Size([1000, 886])


In [6]:
model = MultiModalHouseModel(tabular_input_dim=X_tab_clean.shape[1])

In [7]:
X_img_pt = torch.tensor(X_img, dtype=torch.float32).permute(0, 3, 1, 2)

X_tab_pt = torch.tensor(X_tab_clean, dtype=torch.float32)

y_pt = torch.tensor(y, dtype=torch.float32).view(-1, 1)

input_dim = X_tab_pt.shape[1]
model = MultiModalHouseModel(tabular_input_dim=input_dim)

print(f"Data Ready: {len(X_img_pt)} samples found.")

Data Ready: 1000 samples found.


In [8]:
import torch.optim as optim

criterion = nn.MSELoss() 
optimizer = optim.Adam(model.parameters(), lr=0.001)
epochs = 20

print(f"Starting training on {len(X_img_pt)} samples...")

for epoch in range(epochs):
    model.train()
    optimizer.zero_grad()
    
    predictions = model(X_img_pt, X_tab_pt)
    loss = criterion(predictions, y_pt)
    
    loss.backward()
    optimizer.step()
    
    if (epoch + 1) % 5 == 0:
        mae = torch.mean(torch.abs(predictions - y_pt))
        print(f"Epoch [{epoch+1}/{epochs}] | Loss (MSE): {loss.item():.4f} | MAE: {mae.item():.4f}")

Starting training on 1000 samples...
Epoch [5/20] | Loss (MSE): 790056140800.0000 | MAE: 678209.5625
Epoch [10/20] | Loss (MSE): 790050766848.0000 | MAE: 678206.0625
Epoch [15/20] | Loss (MSE): 790043230208.0000 | MAE: 678201.3750
Epoch [20/20] | Loss (MSE): 790032809984.0000 | MAE: 678194.5625


In [11]:
y_pt_scaled = y_pt / 1_000_000.0

model = MultiModalHouseModel(tabular_input_dim=X_tab_pt.shape[1])
optimizer = optim.Adam(model.parameters(), lr=0.0005) 
criterion = nn.HuberLoss() 

for epoch in range(60):
    model.train()
    optimizer.zero_grad()
    preds = model(X_img_pt, X_tab_pt)
    loss = criterion(preds, y_pt_scaled)
    loss.backward()
    optimizer.step()
    
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1} | Scaled Loss: {loss.item():.6f}")


Epoch 10 | Scaled Loss: 0.173717
Epoch 20 | Scaled Loss: 0.133091
Epoch 30 | Scaled Loss: 0.135045
Epoch 40 | Scaled Loss: 0.127096
Epoch 50 | Scaled Loss: 0.125702
Epoch 60 | Scaled Loss: 0.125116


In [12]:

model.eval()

with torch.no_grad():
    preds_scaled = model(X_img_pt, X_tab_pt)
    
    mae_scaled = torch.mean(torch.abs(preds_scaled - y_pt_scaled)).item()
    mse_scaled = torch.mean((preds_scaled - y_pt_scaled)**2).item()
    rmse_scaled = np.sqrt(mse_scaled)

    actual_mae = mae_scaled * 1_000_000
    actual_mse = mse_scaled * (1_000_000**2)
    actual_rmse = rmse_scaled * 1_000_000

print(f"--- Final Evaluation Metrics ---")
print(f"Mean Absolute Error (MAE):  ${actual_mae:,.2f}")
print(f"Mean Squared Error (MSE):   {actual_mse:,.2f}")
print(f"Root Mean Squared Error (RMSE): ${actual_rmse:,.2f}")

--- Final Evaluation Metrics ---
Mean Absolute Error (MAE):  $398,683.25
Mean Squared Error (MSE):   253,793,060,779.57
Root Mean Squared Error (RMSE): $503,778.78
